## Operational Flow 
__In the notebook we will design the complete operation flow from simple text SQL query to reasoning on related AST instance graph and reconstructing updated SQL text query back based on the reasoning results.__



In [2]:
query = "SELECT account_id, SUM(amount) AS total FROM transactions WHERE account_id = 123 GROUP BY account_id"

### Step-1 : SQL to AST conversion

In [1]:
from sqlglot import parse_one, exp

In [4]:
ast_obj = parse_one(query)

### Step-2 : AST (`sqlgpt_parser`) to AST Tree conversion

In [9]:
import uuid 
from typing import Any, List, Dict, Optional, Type, Tuple


In [10]:
class SQLNode:
    def __init__(self,
                #  id: str = str(uuid.uuid4()),
                 type: str,
                 values: Optional[Any] = None,
                 references: Optional[str] = None,
                 access_state: Optional[str] = None,
                 children: Optional[List['SQLNode']] = None,
                 parent: Optional['SQLNode'] = None
                 ):
        self.id = "n"+str(uuid.uuid4())
        self.type = type
        self.values = values or {}
        self.references = references
        self.access_state = access_state
        self.children = children or []
        self.parent = parent

    def add_child(self, child: 'SQLNode'):
        self.children.append(child)

    def add_parent(self, parent: 'SQLNode'):
        self.parent = parent
    

In [7]:
class AST:
    def __init__(self, root: SQLNode):
        self.root = root 

    # couple of methods need to implement 
    #   1. adding references to Table and Column reference nodes 
    #   2. populate column where there wildcards defined select clause 
    #   3. 


In [8]:
from sqlgpt_parser.parser.tree.literal import (
    BooleanLiteral, DateLiteral, DoubleLiteral, LongLiteral, NullLiteral,
    StringLiteral, TimeLiteral, TimestampLiteral, DefaultLiteral, ErrorLiteral,
)
from sqlgpt_parser.parser.tree.expression import QualifiedNameReference, FunctionCall
from sqlgpt_parser.parser.tree.table import Table

In [ ]:
class F2toF3Converter:
    def __init__(self):
        self.mapping: List[Tuple[Tuple[Type, ...], str, Dict[str, str]]] = [
            (
                (BooleanLiteral, DateLiteral, DoubleLiteral, LongLiteral, NullLiteral,
                 StringLiteral, TimeLiteral, TimestampLiteral, DefaultLiteral, ErrorLiteral),
                'Literal',
                {'value': 'value'}
            ),
            (
                (QualifiedNameReference,),
                'ColmnRef',
                {'name': 'name'}
            ),
            (
                (FunctionCall,),
                'Function',
                {'name': 'name'}
            ),
            (
                (Table,),
                'TableRef',
                {'name': 'name'}
            )
        ]

    def convert(self, node: Any, parent= Optional[SQLNode]) -> SQLNode:
        node_type = type(node)

        for types, target_type, attr_map in self.mapping:
            if isinstance(node, types):
                values = {k: str(getattr(node, v)) for k, v in attr_map.items()}
                sql_node = SQLNode(type=target_type, values=values, parent=parent)
            
                    